# UQ Evaluation Metrics

Comprehensive evaluation of uncertainty quality.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.metrics import PICP, NLL, CRPS, IntervalScore, AUROC, FPR95, AURC
from deepuq.models import MLP
from deepuq.methods import DeepEnsemble

## Setup: Train Model with UQ

In [ ]:
# Generate data
np.random.seed(42)
torch.manual_seed(42)

X = np.linspace(-3, 3, 200).reshape(-1, 1)
y = np.sin(X) + 0.1 * X**2 + np.random.randn(*X.shape) * 0.1

X_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(y, dtype=torch.float32)

# Train ensemble of 5 MLPs
ensemble = DeepEnsemble(
    base_model=MLP,
    model_kwargs={"input_dim": 1, "hidden_dims": [64, 64], "output_dim": 1},
    n_members=5,
)
ensemble.fit(X_train, y_train, epochs=200, lr=1e-3)

# Get predictions
X_test = torch.tensor(np.linspace(-3, 3, 100).reshape(-1, 1), dtype=torch.float32)
y_test = torch.tensor(np.sin(X_test.numpy()) + 0.1 * X_test.numpy()**2 + np.random.randn(100, 1) * 0.1, dtype=torch.float32)

mean, std = ensemble.predict_uq(X_test)
print(f"Predictions: mean shape={mean.shape}, std shape={std.shape}")

## Calibration Metrics

In [ ]:
# Compute PICP at different confidence levels
confidence_levels = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
picp_values = []

for alpha in confidence_levels:
    picp = PICP(mean, std, y_test, confidence=alpha)
    picp_values.append(picp)

# Plot calibration curve
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.plot(confidence_levels, picp_values, "bo-", label="Model")
plt.xlabel("Expected coverage")
plt.ylabel("Observed coverage (PICP)")
plt.title("Calibration Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Calibration results:")
for alpha, picp in zip(confidence_levels, picp_values):
    print(f"  {alpha*100:.0f}% interval: PICP = {picp:.3f}")

## Scoring Rules

In [ ]:
# Compute scoring rules
nll = NLL(mean, std, y_test)
crps = CRPS(mean, std, y_test)
interval_score = IntervalScore(mean, std, y_test, confidence=0.9)

print("Scoring Rules:")
print(f"{'Metric':<20} {'Value':<10}")
print("-" * 30)
print(f"{'NLL':<20} {nll:.4f}")
print(f"{'CRPS':<20} {crps:.4f}")
print(f"{'Interval Score (90%)':<20} {interval_score:.4f}")

## OOD Detection

In [ ]:
# In-distribution data
X_id = torch.tensor(np.random.uniform(-3, 3, (100, 1)), dtype=torch.float32)
mean_id, std_id = ensemble.predict_uq(X_id)

# OOD data (outside training range)
X_ood = torch.tensor(np.random.uniform(5, 8, (100, 1)), dtype=torch.float32)
mean_ood, std_ood = ensemble.predict_uq(X_ood)

# Use uncertainty as OOD score
scores_id = std_id.numpy().ravel()
scores_ood = std_ood.numpy().ravel()

# Compute OOD detection metrics
auroc = AUROC(scores_id, scores_ood)
fpr95 = FPR95(scores_id, scores_ood)

print(f"OOD Detection Results:")
print(f"  AUROC: {auroc:.4f}")
print(f"  FPR@95% TPR: {fpr95:.4f}")

# Visualize score distributions
plt.figure(figsize=(8, 4))
plt.hist(scores_id, bins=30, alpha=0.5, label="In-distribution", density=True)
plt.hist(scores_ood, bins=30, alpha=0.5, label="OOD", density=True)
plt.xlabel("Uncertainty (std)")
plt.ylabel("Density")
plt.title("Uncertainty Scores: ID vs OOD")
plt.legend()
plt.tight_layout()
plt.show()

## Selective Prediction Metrics

In [ ]:
# Compute AURC and plot risk-coverage curve
aurc, coverages, risks = AURC(mean, std, y_test, return_curve=True)

print(f"AURC: {aurc:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(coverages, risks, "b-", linewidth=2)
plt.xlabel("Coverage")
plt.ylabel("Risk (MSE)")
plt.title("Risk-Coverage Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("
Selective prediction: rejecting uncertain predictions reduces risk")
print(f"  Risk at 100% coverage: {risks[-1]:.4f}")
print(f"  Risk at 80% coverage: {risks[int(0.8*len(risks))]:.4f}")
print(f"  Risk at 50% coverage: {risks[int(0.5*len(risks))]:.4f}")